# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Provisional lane: Lane 2 — Refresh / Content Opportunity Scoring.**

I'm picking this lane because it produces the artifact I actually want to end the internship with: a ranked
review queue, not just a chart. The lane guide's own starter pipeline already shows a learned ranking beating
a hand-written rule on this exact dataset (baseline Precision@50 = 0.240 vs. random forest Precision@50 = 0.740,
from `outputs/model_report.md`) — so there's already evidence that "which page first" is a question worth more
than an if-statement here. The other lanes are interesting (signal analysis, clustering, CTR-only scoring) but
they mostly explain *why* something is happening; this lane forces me to also decide *what to do about it and
in what order*, which is the harder and more useful problem. I'm keeping it provisional: if the signal audit in
Week 4 shows the label is too blunt to support ranking (see Section 4), I may narrow to a CTR-only slice or
pivot toward the Growth/Recovery freestyle direction with a proper future-window label instead of `trend_direction`.


## 2. The question: decision, action, cost of a wrong call

**Search question:** Given a client's existing content inventory, which pages should a content reviewer look at
first this week, out of far more pages than they have time to check by hand?

**Unit of analysis:** one content item (one row = one pseudonymized page, `content_id`), scored and ranked within
its client (`client_id`) — never compared to pages belonging to a different client, since exposure and competition
differ by site.

**Output:** a ranked queue of content items per client, each with a numeric priority score, a plain-language
reason code (e.g. "declining with demand," "page-one decay risk," "low CTR at a good position"), and a suggested
action (refresh, rewrite title/meta, monitor, or leave alone).

**Decision this improves:** which handful of pages a content editor spends their limited hours on this cycle,
instead of triaging by gut feel or working strictly newest/oldest first.

**Who acts, and how:** a content editor or SEO strategist opens the top of the queue and works down it; the
action they take (rewrite, expand, refresh metadata, or leave it alone) depends on the reason code attached to
each row.

**Cost of a wrong call:** two different failure directions, and they cost differently:
- *False positive* (the model flags a page as worth reviewing, but it wasn't really a priority): wasted editor
  hours — the editor spends 30-60 minutes reviewing/refreshing a page that didn't need it, and a genuinely
  weak page further down the queue waits another cycle.
- *False negative* (a genuinely declining or high-opportunity page never surfaces near the top): the real cost —
  continued organic traffic loss on a page nobody looks at, compounding for as long as it stays unreviewed.
Because editor time is the scarce resource and traffic loss compounds silently, I'll weight recall on the
truly high-value pages more heavily than raw accuracy, and evaluate with Precision@K / Recall@K rather than a
single global accuracy number — matching how the queue is actually consumed (top-down, K at a time).

**Why data or ML helps at all:** a single hand-written rule can catch one pattern (e.g. "stale and still visible")
but a page can be a priority for several tangled, correlated reasons at once — declining trend *and* thin content
*and* okay position but weak CTR — and the *relative* ranking across those overlapping signals is exactly the
kind of pattern that's real but too messy to hand-code confidently. That said, I'm not assuming ML is required
before I've checked: Section 3 below already shows a hand-written baseline captures a meaningful chunk of
pages, so part of the work is honestly quantifying how much a learned model adds *over* that baseline, not
just assuming it wins.


## 3. Quick look at the data (2-3 real numbers)

*Loading the starter CSV and pulling numbers that speak directly to the lane choice above: how big the candidate pool is, and whether different reason codes pick out different pages (i.e. there's real structure to rank, not one obvious rule).*

### Findings

- The starter dataset contains **30,000 rows**, **44 columns**, and data from **32 clients**.
- After applying the eligibility filter (`impressions_90d > 0` and `content_age_days >= 90`), **30,000 pages** remain eligible for scoring.
- **54.2%** of eligible pages have `trend_direction = "down"`, indicating that declining performance is common enough to justify prioritization.
- The **declining_with_demand** reason code fires on **13,152 pages (43.8%)**.
- The **page_one_decay_risk** reason code fires on **7,076 pages (23.6%)**.
- **2,827 pages** trigger both signals simultaneously, suggesting that the signals capture related but distinct opportunities and supporting the use of a ranking model rather than a single hard rule.


In [1]:
import pandas as pd

df = pd.read_csv("content_refresh_anonymized(2).csv")

print(f"Starter dataset: {df.shape[0]:,} rows x {df.shape[1]} columns, {df['client_id'].nunique()} clients")

# Apply the same eligibility filter the starter pipeline uses, so these numbers
# describe the population my ranking would actually run over.
elig = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].drop_duplicates("content_id")
print(f"Eligible rows (impressions_90d > 0, content_age_days >= 90): {len(elig):,}")

# Real number 1: how much of the inventory the current proxy label flags as "declining"
declining_rate = (elig["trend_direction"] == "down").mean()
print(f"\nShare of eligible pages with trend_direction == 'down': {declining_rate:.1%}")

# Real number 2: one concrete, actionable baseline reason code and its volume
declining_with_demand = ((elig["trend_direction"] == "down") & (elig["impressions_90d"] >= 100)).sum()
print(f"'declining_with_demand' reason code fires on: {declining_with_demand:,} pages "
      f"({declining_with_demand/len(elig):.1%} of eligible)")

# Real number 3: a second, different reason code -> shows overlapping-but-distinct signals,
# which is exactly the "why ML might help" argument from Section 2.
page_one_decay = ((elig["avg_position"] > 0) & (elig["avg_position"] <= 10) & (elig["content_age_days"] >= 180)).sum()
print(f"'page_one_decay_risk' reason code fires on: {page_one_decay:,} pages "
      f"({page_one_decay/len(elig):.1%} of eligible)")

overlap = (
    (elig["trend_direction"] == "down") & (elig["impressions_90d"] >= 100)
    & (elig["avg_position"] > 0) & (elig["avg_position"] <= 10) & (elig["content_age_days"] >= 180)
).sum()
print(f"Pages triggering BOTH reason codes at once: {overlap:,} "
      f"-- these two signals are not redundant, they pick out different (and sometimes overlapping) pages.")


Starter dataset: 30,000 rows x 44 columns, 32 clients
Eligible rows (impressions_90d > 0, content_age_days >= 90): 30,000

Share of eligible pages with trend_direction == 'down': 54.2%
'declining_with_demand' reason code fires on: 13,152 pages (43.8% of eligible)
'page_one_decay_risk' reason code fires on: 7,076 pages (23.6% of eligible)
Pages triggering BOTH reason codes at once: 2,827 -- these two signals are not redundant, they pick out different (and sometimes overlapping) pages.


## 4. Careful words: what I can and can't claim

**What this work CAN say, if it holds up through validation:**
- *Observed*: which pages in this dataset show declining trend, thin content, page-one decay risk, or weak
  CTR-for-position, measured from logged search/analytics signals.
- *Directional / decision-support*: a ranking that puts pages more likely to be worth a review nearer the top
  of the queue than a plain rule does, evaluated with Precision@K on held-out clients.
- A comparison of "how much better does a learned model rank pages than the hand-written baseline," honestly
  reported even if the answer is "not much."

**What this work will NEVER claim, and I need to keep saying that out loud:**
- *Not causal.* Even if a page is correctly flagged and later recovers after a refresh, I have no experiment
  (no A/B test, no causal design) proving the refresh caused the recovery — recovery could be consolidation,
  seasonality, or a sibling page's change instead.
- *Not a Google algorithm claim.* Nothing here proves what any search engine's ranking algorithm does; it only
  describes observed impressions, clicks, and position in this data.
- *Not proof the model beats humans.* At best it shows the model beats a specific, simple written rule on this
  specific 30k-row anonymized slice, with client-holdout validation — not that it would generalize unchanged to
  the full ~79M-row warehouse or to a different client base.
- **The label caveat that matters most for this lane:** the starter `trend_direction` field (and therefore
  `is_declining_label`) is a *bucket computed from the current 90-day window*, not a future observed outcome.
  A stronger version of this lane — the one I want to move toward by Week 4-5 — redefines the label as
  "declined over the NEXT 30 days, given the PRIOR 90 days of features," using the warehouse's daily fact table.
  Until I've rebuilt that, any "the model predicts decline" language in this notebook really means "the model
  reproduces the current-window trend bucket," which is a weaker and more honest thing to say.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
